## Prepare ADRD rates dataset

In [1]:
## Load packages ----
import numpy as np
import pandas as pd
import sshtunnel
import psycopg2 as pg
import os

import json
import sys

import seaborn as sns
import matplotlib.pyplot as plt

/n/home_fasse/maudirac/.conda/envs/medicare_QC/lib/python3.9/site-packages/paramiko/transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [2]:
## read zip to county crosswalk ----
zip_to_county = pd.read_csv('../data/input/shared_data/zip_county_2010.csv')
zip_to_county = zip_to_county[['ZIP', 'COUNTY']]
zip_to_county = zip_to_county.rename(columns = {'ZIP':'zip', 'COUNTY':'county'})
zip_w = zip_to_county.groupby(['zip'])['county'].count().reset_index()
zip_w = zip_w.rename(columns = {'county':'w'})
zip_w['w'] = 1 / zip_w.w
zip_to_county = zip_to_county.merge(zip_w)
zip_to_county.w.describe()

count    46875.000000
mean         0.775168
std          0.278345
min          0.166667
25%          0.500000
50%          1.000000
75%          1.000000
max          1.000000
Name: w, dtype: float64

In [3]:
## Open ssh tunnel to DB host ----
tunnel = sshtunnel.SSHTunnelForwarder(
    ('nsaph.rc.fas.harvard.edu', 22),
    ssh_username=f'{os.environ["MY_NSAPH_SSH_USERNAME"]}',
    ssh_private_key=f'{os.environ["HOME"]}/.ssh/id_rsa', 
    ssh_password=f'{os.environ["MY_NSAPH_SSH_PASSWORD"]}', 
    remote_bind_address=("localhost", 5432)
)

tunnel.start()

## Open connection to DB ----
connection = pg.connect(
    host='localhost',
    database='nsaph2',
    user=f'{os.environ["MY_NSAPH_DB_USERNAME"]}',
    password=f'{os.environ["MY_NSAPH_DB_PASSWORD"]}', 
    port=tunnel.local_bind_port
)

In [4]:
## define functions ----
def get_outcomes(read_path):
    """ Get and return ICD codes """""
    f = open(read_path)
    res_dict = json.load(f)
    f.close()
    res_dict = json.loads(res_dict[0])
    return res_dict

def get_outcomes_set(outcome=None, year=None):
    """ Uses ICD9 for years prior 2015 and ICD10 otherwise """
    if year < 2015:
        outcomes_set = outcomes[outcome]["icd9"]
    elif year > 2015:
        outcomes_set = outcomes[outcome]["icd10"]
    else:
        outcomes_set = outcomes[outcome]["icd10"] + \
                       outcomes[outcome]["icd9"]
    return set(outcomes_set)

def get_outcome_in_diagnoses(outcomes_set=None, diagnoses=None):
    return any(o_ in outcomes_set for o_ in diagnoses)

In [5]:
years_ = [y_ for y_ in range(2000, 2019)]
years_.remove(2015)
years_.remove(2006)

## bene_county_df

In [6]:
bene_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = f"""
    SELECT 
        zip,
        year,
        race, 
        sex,
        case 
            when age < 65 then '<65'
            when age >= 65 and age < 75 then '[65,75)'
            when age >= 75 and age < 85 then '[75,85)'
            when age >= 85 then '>85'
        end age_grp,
        count(*) as n_enrollees
    FROM 
        medicare.enrollments
        LEFT JOIN medicare.beneficiaries ON medicare.enrollments.bene_id = medicare.beneficiaries.bene_id
    WHERE 
        year in ('{y_}') AND
        state = 'NC' AND 
        race in ('1', '2') AND
        sex in ('1', '2')
    GROUP BY 
        age_grp, 
        year, 
        zip, 
        race, 
        sex
    ;
    """
    ## Request query ----
    %time b = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    bene_zip_list.append(b)

2000
CPU times: user 325 ms, sys: 51.4 ms, total: 376 ms
Wall time: 6.56 s
2001
CPU times: user 105 ms, sys: 25.1 ms, total: 130 ms
Wall time: 3.91 s
2002
CPU times: user 107 ms, sys: 18.6 ms, total: 126 ms
Wall time: 3.99 s
2003
CPU times: user 99.9 ms, sys: 11.5 ms, total: 111 ms
Wall time: 4.55 s
2004
CPU times: user 90.8 ms, sys: 21.4 ms, total: 112 ms
Wall time: 4.29 s
2005
CPU times: user 93.8 ms, sys: 17.3 ms, total: 111 ms
Wall time: 4.23 s
2007
CPU times: user 95.7 ms, sys: 17.5 ms, total: 113 ms
Wall time: 6.12 s
2008
CPU times: user 97.8 ms, sys: 16.4 ms, total: 114 ms
Wall time: 4.89 s
2009
CPU times: user 100 ms, sys: 13.7 ms, total: 114 ms
Wall time: 5.09 s
2010
CPU times: user 98.4 ms, sys: 16.8 ms, total: 115 ms
Wall time: 5.08 s
2011
CPU times: user 89.6 ms, sys: 23.6 ms, total: 113 ms
Wall time: 5.32 s
2012
CPU times: user 93.2 ms, sys: 21.7 ms, total: 115 ms
Wall time: 9.54 s
2013
CPU times: user 97.3 ms, sys: 19 ms, total: 116 ms
Wall time: 9.7 s
2014
CPU times: use

In [7]:
bene_zip_df = pd.concat(bene_zip_list)
bene_county_df = bene_zip_df.merge(zip_to_county)
bene_county_df['n_enrollees'] = bene_county_df.n_enrollees * bene_county_df.w
bene_county_df = bene_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_enrollees'].sum().reset_index()

In [8]:
# pct crosswalked 
bene_zip_df.n_enrollees.sum()

25491220

In [9]:
bene_county_df.n_enrollees.sum()

24320746.0

## adrd_county_df

In [10]:
adm_zip_list = list()

for y_ in years_: 
    print(y_)
    
    ## Define query ----
    sql_query = """
    SELECT
        bene.bene_id,
        diagnoses,
        zip,
        year,
        race,
        sex, 
        EXTRACT(YEAR FROM dob) as yob_
    FROM 
        medicare.beneficiaries as bene
    RIGHT JOIN (
        SELECT 
            bene_id, 
            diagnoses, 
            zip, 
            year
        FROM 
            medicare.admissions as adm
        WHERE
            year in ('2000') AND
            state = 'NC'
    ) as adm
    ON bene.bene_id = adm.bene_id
    WHERE
      race in ('1', '2') AND
      sex in ('1', '2')
    ;
    """
    ## Request query ----
    %time a = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()
    adm_zip_list.append(a)

2000
CPU times: user 5.32 s, sys: 1.4 s, total: 6.73 s
Wall time: 5.72 s
2001
CPU times: user 7.12 s, sys: 1.53 s, total: 8.65 s
Wall time: 7.43 s
2002
CPU times: user 6.6 s, sys: 1.53 s, total: 8.13 s
Wall time: 6.94 s
2003
CPU times: user 6.84 s, sys: 1.5 s, total: 8.35 s
Wall time: 7.09 s
2004
CPU times: user 7.71 s, sys: 1.51 s, total: 9.21 s
Wall time: 8.01 s
2005
CPU times: user 6.46 s, sys: 1.53 s, total: 7.99 s
Wall time: 6.78 s
2007
CPU times: user 6.62 s, sys: 1.37 s, total: 7.99 s
Wall time: 6.82 s
2008
CPU times: user 7.04 s, sys: 1.46 s, total: 8.5 s
Wall time: 7.33 s
2009
CPU times: user 7.47 s, sys: 1.49 s, total: 8.96 s
Wall time: 7.75 s
2010
CPU times: user 7.72 s, sys: 1.54 s, total: 9.26 s
Wall time: 8.02 s
2011
CPU times: user 8.04 s, sys: 1.52 s, total: 9.55 s
Wall time: 8.35 s
2012
CPU times: user 8.56 s, sys: 1.57 s, total: 10.1 s
Wall time: 8.88 s
2013
CPU times: user 5.12 s, sys: 1.4 s, total: 6.52 s
Wall time: 5.34 s
2014
CPU times: user 9.14 s, sys: 1.51 s, t

In [11]:
adm_zip_df = pd.concat(adm_zip_list)
adm_zip_df['age'] = adm_zip_df.year - adm_zip_df.yob_
adm_zip_df['age_grp'] = pd.cut(x=adm_zip_df['age'], 
                               bins=[min(adm_zip_df.age), 65, 75, 85, max(adm_zip_df.age)],
                               labels=['<65', '[65,75)', '[75,85)', '>85'])

## read outcomes ----
read_path = '../data/input/shared_data/icd_codes.json'
outcomes = get_outcomes(read_path)

## find diagnoses ----
for outcome in ['adrd']:
    adm_zip_df[outcome] = [get_outcome_in_diagnoses(get_outcomes_set(outcome, y_), d_[:1]) for y_, d_ in zip(adm_zip_df.year, adm_zip_df.diagnoses)]

keep = adm_zip_df[['adrd']].any(axis=1)
adm_zip_df = adm_zip_df[keep]
adm_zip_df = adm_zip_df.drop(columns='adrd')
adm_zip_df = adm_zip_df.groupby(['year', 'zip', 'race', 'sex', 'age_grp'])['bene_id'].count().reset_index()
adm_zip_df = adm_zip_df.rename(columns = {'bene_id':'n_adrd'})
adm_county_df = adm_zip_df.merge(zip_to_county)
adm_county_df['n_adrd'] = adm_county_df.n_adrd * adm_county_df.w
adm_county_df = adm_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'])['n_adrd'].sum().reset_index()

In [12]:
adm_zip_df.n_adrd.sum()

46733

In [13]:
adm_county_df.n_adrd.sum()

45288.0

## adrd_df

In [14]:
county_ = sorted(bene_county_df.county.unique())
year_ = sorted(bene_county_df.year.unique())
race_ = sorted(bene_county_df.race.unique())
sex_ = sorted(bene_county_df.sex.unique())
age_grp_ = sorted(bene_county_df.age_grp.unique())

In [15]:
len(county_)

1125

In [16]:
adrd_df = pd.DataFrame({'county':county_}).merge(pd.DataFrame({'year':year_}), how = 'cross')
adrd_df = adrd_df.merge(pd.DataFrame({'race':race_}), how = 'cross')
adrd_df = adrd_df.merge(pd.DataFrame({'sex':sex_}), how = 'cross')
adrd_df = adrd_df.merge(pd.DataFrame({'age_grp':age_grp_}), how = 'cross')

In [17]:
adrd_df['state'] = [str(x)[0:2] for x in adrd_df.county]
adrd_df = adrd_df[adrd_df.state == '37']

In [18]:
len(adrd_df.county.unique())

100

In [19]:
adrd_df = adrd_df.merge(bene_county_df, how = 'left')
adrd_df = adrd_df.merge(adm_county_df, how = 'left')

In [20]:
adrd_df.n_enrollees.sum()

24311184.0

In [21]:
adrd_df.n_adrd.sum()

45288.0

In [22]:
adrd_df.n_enrollees.isnull().mean()

0.010073529411764705

In [23]:
adrd_df.n_adrd.isnull().mean()

0.9411764705882353

In [24]:
adrd_df.to_csv("../data/intermediate/adrd.csv")